In [1]:
import pyNUISANCE as pn

In [5]:
input_vectors = {
    -14: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_numubar.hepmc3.gz"),
    -12: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_nuebar.hepmc3.gz"),
    12: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_nue.hepmc3.gz"),
    14: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_numu.hepmc3.gz")
}

for nupdg, evs in input_vectors.items():
    if not evs:
        raise RuntimeError(f"Failed to read event file for {nupdg}")

In [6]:
from pyProSelecta import event, part, unit, pdg, p3mod, momentum, kinetic_energy, ext
from pyNUISANCE import nhm
import numpy as np

def GetPrimaryLepton(ev):
    beam_part = nhm.EventUtils.GetBeamParticle(ev)
    prim_vertex = nhm.EventUtils.GetPrimaryVertex(ev)
    for part in prim_vertex.particles_out():
        if (part.pid() == beam_part.pid()) or\
           ( (abs(part.pid()) + 1 ) == abs(beam_part.pid())):
          return part
    raise runtime_error("Failed to find primary lepton out: %s -> [%s]" % \
                        (beam_part.pid(), ",".join(part.pid for part in prim_vertex.particles_out())))

def PrimaryLeptonTotalEnergy(ev):
    pl = GetPrimaryLepton(ev)
    return (pl.momentum().e() / unit.GeV_c) if pl.pid() % 2 else 0

def NeutralPionTotalEnergy(ev):
    return part.sum(momentum, event.all_out_part(ev, 111)).e() / unit.GeV_c

def NumberOfChargedPions(ev):
    return event.num_out_part(ev, [211,-211], flatten=True)

def ChargedPionKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, [211, -211], flatten=True)) / unit.GeV_c

def ProtonKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, 2212)) / unit.GeV_c

def ReconstructedNeutrinoEnergy_true(ev):
    return PrimaryLeptonTotalEnergy(ev) + ChargedPionKineticEnergy(ev) + (NumberOfChargedPions(ev)*0.1395) +  NeutralPionTotalEnergy(ev) + ProtonKineticEnergy(ev)

event_frames = {}

for nupdg, evs in input_vectors.items():
    event_frames[nupdg] = pn.EventFrameGen(evs) \
                            .add_column("pid_nu", lambda ev: nhm.EventUtils.GetBeamParticle(ev).pid()) \
                            .add_column("E_nu", ext.enu_GeV) \
                            .add_column("is_CC", ext.isCC) \
                            .add_column("pid_lep", lambda ev: GetPrimaryLepton(ev).pid()) \
                            .add_column("E_lepton", PrimaryLeptonTotalEnergy) \
                            .add_column("T_proton", ProtonKineticEnergy) \
                            .add_column("num_pi0", lambda ev: event.num_out_part(ev, 111)) \
                            .add_column("E_pi0", NeutralPionTotalEnergy) \
                            .add_column("num_cpi", NumberOfChargedPions) \
                            .add_column("T_cpi", ChargedPionKineticEnergy) \
                            .add_column("E_nu_rec_true", ReconstructedNeutrinoEnergy_true).all()

output_columns = [
    "pid_nu",
    "E_nu",
    "is_CC",
    "pid_lep",
    "E_lepton",
    "T_proton",
    "num_pi0",
    "E_pi0",
    "num_cpi",
    "T_cpi",
    "E_nu_rec_true" ]

print(event_frames[14])

 --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 | event.number | weight.cv | fatx_per_su$ | fatx_per_su$ | process.id | pid_nu |   E_nu | is_CC | pid_lep | E_lepton | T_proton | num_pi0 | E_pi0 | num_cpi |   T_cpi | E_nu_rec_tr$ |
 --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 |            0 |         1 |    1.374e-06 |    3.434e-08 |        500 |     14 |  3.222 |     1 |      13 |    2.019 |   0.3915 |       0 |     0 |       2 |  0.5405 |         3.23 |
 |            1 |         1 |    6.869e-07 |    1.717e-08 |        251 |     14 |    3.2 |     0 |      14 |        0 |        0 |       0 |     0 |       0 |       0 |            0 |
 |            2 |         1 |    4.579e-07 |    1.145e-08 |        200 |     14 

In [7]:
import pandas as pa

def todf(ef, index):
  df = pa.DataFrame(data=ef.table,
                    index=index,
                    columns=ef.column_names)
  for row in ["event.number", "process.id", "pid_nu", "is_CC", "pid_lep", "num_pi0", "num_cpi"]:
    df[row] = pa.to_numeric(df[row], downcast="integer")
  return df

In [17]:
df_numu = todf(event_frames[14], [i for i in range(event_frames[14].table.shape[0])])
df_nue = todf(event_frames[12], [i + event_frames[14].table.shape[0] for i in range(event_frames[12].table.shape[0])])
df_numode = pa.concat([df_numu, df_nue])
df_numode = df_numode.sample(frac=1)
df_numode

,event.number,weight.cv,fatx_per_sumw.pb_per_target.estimate,fatx_per_sumw.pb_per_nucleon.estimate,process.id,pid_nu,E_nu,is_CC,pid_lep,E_lepton,T_proton,num_pi0,E_pi0,num_cpi,T_cpi,E_nu_rec_true
59738,59738,1.0,2.299594e-11,5.748985e-13,200,14,3.846146,1,13,3.763478,0.081471,0,0.000000,0,0.000000,3.844949
158084,58084,1.0,2.407155e-11,6.017886e-13,401,12,2.950922,1,11,1.002610,1.584322,1,0.370516,0,0.000000,2.957448
32198,32198,1.0,4.266451e-11,1.066613e-12,400,14,3.274167,1,13,1.852132,0.021587,0,0.000000,1,0.116756,2.129975
161645,61645,1.0,2.268105e-11,5.670261e-13,400,12,1.022683,1,11,0.485148,0.248243,0,0.000000,1,0.137326,1.010216
54457,54457,1.0,2.522594e-11,6.306486e-13,550,14,4.264277,0,14,0.000000,0.000000,1,0.872509,1,0.541744,1.553753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99182,99182,1.0,1.385071e-11,3.462676e-13,200,14,1.927738,1,13,1.878242,0.010028,0,0.000000,0,0.000000,1.888270
18774,18774,1.0,7.316935e-11,1.829234e-12,600,14,12.055444,1,13,4.129656,0.312088,1,0.417384,4,4.427281,9.844410
122037,22037,1.0,6.344477e-11,1.586119e-12,600,12,17.780477,1,11,9.970583,0.000000,1,1.301989,3,5.058588,16.749660
88162,88162,1.0,1.558198e-11,3.895496e-13,601,14,3.465025,0,14,0.000000,0.129575,0,0.000000,0,0.000000,0.129575


In [18]:
df_numode.to_csv("neutrino_mode_events.csv",index_label="event_num",columns=output_columns, float_format="%.4f")

In [19]:
df_numubar = todf(event_frames[-14], [i for i in range(event_frames[-14].table.shape[0])])
df_nuebar = todf(event_frames[-12], [i + event_frames[-14].table.shape[0] for i in range(event_frames[12].table.shape[0])])
df_nubarmode = pa.concat([df_numubar, df_nuebar])
df_nubarmode = df_nubarmode.sample(frac=1).reset_index(drop=True)
df_nubarmode

,event.number,weight.cv,fatx_per_sumw.pb_per_target.estimate,fatx_per_sumw.pb_per_nucleon.estimate,process.id,pid_nu,E_nu,is_CC,pid_lep,E_lepton,T_proton,num_pi0,E_pi0,num_cpi,T_cpi,E_nu_rec_true
0,10328,1.0,5.177334e-11,1.294334e-12,426,-14,3.168285,1,-13,2.396825,0.155727,0,0.000000,1,0.073080,2.765132
1,38768,1.0,1.379367e-11,3.448418e-13,427,-14,2.975573,1,-13,2.194425,0.231804,0,0.000000,1,0.415057,2.980786
2,30912,1.0,1.729909e-11,4.324773e-13,276,-14,4.699363,0,-14,0.000000,0.000000,0,0.000000,0,0.000000,0.000000
3,97700,1.0,5.473504e-12,1.368376e-13,625,-14,23.250740,1,-13,21.065681,0.000000,0,0.000000,2,1.567396,22.912077
4,9967,1.0,5.456620e-11,1.364155e-12,325,-12,0.970417,1,-11,0.212908,0.666411,0,0.000000,0,0.000000,0.879319
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,3334,1.0,1.630932e-10,4.077331e-12,225,-12,4.228897,1,-11,3.931455,0.064100,0,0.000000,0,0.000000,3.995556
199996,79534,1.0,6.723667e-12,1.680917e-13,675,-14,12.643320,0,-14,0.000000,0.224524,0,0.000000,0,0.000000,0.224524
199997,12854,1.0,4.231162e-11,1.057791e-12,225,-12,2.163271,1,-11,2.060095,0.000000,0,0.000000,0,0.000000,2.060095
199998,78566,1.0,6.806507e-12,1.701627e-13,575,-14,1.853330,0,-14,0.000000,0.013456,1,0.162346,2,0.735647,1.190448


In [20]:
df_nubarmode.to_csv("antineutrino_mode_events.csv",index_label="event_num",columns=output_columns, float_format="%.4f")